# 07 — Универсальный Kaggle submission

Этот notebook восстанавливает **одного уже измеренного кандидата**, обучает его на всём train и создаёт проверенный inference-файл. Он не выполняет CV и не меняет champion.


## Правила

1. Выбор делается устойчивым Candidate ID, а не свободной комбинацией признаков и estimator.
2. EXP со статусом reject разрешён, если его feature hook проходит inference audit.
3. Модель из CV не переиспользуется: pipeline заново fit на всём train.
4. Запись CSV, model artifact и SUB-карточки выполняется только последней ячейкой.


In [ ]:
from pathlib import Path
import importlib
import sys

from IPython.display import display

CURRENT_DIR = Path.cwd().resolve()
PROJECT_ROOT = next(
    candidate for candidate in (CURRENT_DIR, *CURRENT_DIR.parents)
    if (candidate / 'README.md').exists() and (candidate / 'src').exists()
)
SRC_DIR = PROJECT_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from ml_project import DataCatalog
import ml_project.config as project_config
import ml_project.baseline_config as baseline_config
import ml_project.model_screening_config as screening_config
import ml_project.modeling as modeling_tools
import ml_project.submission as submission_tools

print(f'Корень проекта: {PROJECT_ROOT}')


## 1. Каталог доступных кандидатов

В таблицу автоматически попадают baseline, сохранённые EXP-кандидаты и все модели из MS-screening. Статус reject не скрывает кандидата.


In [ ]:
project_config = importlib.reload(project_config)
baseline_config = importlib.reload(baseline_config)
screening_config = importlib.reload(screening_config)
submission_tools = importlib.reload(submission_tools)

initial_settings = baseline_config.BASELINE
MODEL_GROUPS = screening_config.MODEL_GROUPS
catalog = submission_tools.candidate_catalog(
    PROJECT_ROOT,
    baseline_model_label=modeling_tools.resolved_model_name(initial_settings),
)
display(submission_tools.candidate_catalog_report(catalog))


## 2. Выбор submission

Редактируйте только следующую ячейку. Для нового файла всегда используйте новый `SUB-ID`.


In [ ]:
SUBMISSION_ID = "SUB-001"
SELECTED_CANDIDATE = "EXP-001/model"

selected = catalog[catalog['candidate_id'].eq(SELECTED_CANDIDATE)]
if selected.empty:
    raise KeyError(f'Кандидат не найден: {SELECTED_CANDIDATE}')
display(submission_tools.candidate_catalog_report(selected))


## 3. Данные и версии

Sample submission используется, если файл настроен и существует. Без него notebook построит двухколоночный файл из KEY и TARGET.


In [ ]:
DATASETS = project_config.DATASETS
FEATURE_GROUPS = project_config.FEATURE_GROUPS
KEY = project_config.KEY
RAW_DIR = project_config.RAW_DIR
TARGET = project_config.TARGET
TRAIN_DATASET = project_config.TRAIN_DATASET
INFERENCE_DATASET = project_config.INFERENCE_DATASET

files = DataCatalog(PROJECT_ROOT, RAW_DIR, DATASETS)
files.validate()
train = files.load(TRAIN_DATASET)
inference = files.load(INFERENCE_DATASET)
available = set(files.available_names())
sample_dataset = next(
    (
        name for name, spec in DATASETS.items()
        if spec.get('role') == 'submission_example' and name in available
    ),
    None,
)
sample_submission = files.load(sample_dataset) if sample_dataset else None
file_report = files.file_report().set_index('dataset')
dataset_versions = {
    'train': str(file_report.loc[TRAIN_DATASET, 'sha256']),
    'inference': str(file_report.loc[INFERENCE_DATASET, 'sha256']),
}
print('Train:', train.shape)
print('Inference:', inference.shape)
print('Sample submission:', sample_dataset or 'не найден — будет создан по контракту')


## 4. Восстановление и inference audit

До fit notebook показывает точный source, feature set, estimator и состояние кода. Контекстно-зависимые raw-признаки будут перечислены в audit и заблокируют predict.


In [ ]:
prepared = submission_tools.prepare_submission_candidate(
    PROJECT_ROOT,
    SELECTED_CANDIDATE,
    train,
    inference,
    FEATURE_GROUPS,
    target=TARGET,
    key=KEY,
    initial_settings=initial_settings,
    model_groups=MODEL_GROUPS,
)
display(submission_tools.submission_contract_report(prepared))
if prepared.inference_audit.empty:
    print('Inference audit: passed')
else:
    display(prepared.inference_audit)


## 5. Full-train fit и predict

Эта ячейка ещё ничего не сохраняет. Проверьте validation, распределение и первые/последние строки.


In [ ]:
prediction = submission_tools.fit_submission_candidate(
    prepared,
    sample_submission,
    key=KEY,
    target=TARGET,
)
display(prediction.validation)
display(prediction.prediction_distribution)
display(prediction.frame.head())
display(prediction.frame.tail())


## 6. Явно сохранить официальный SUB-run

После визуальной проверки выполните последнюю ячейку. CSV и fitted pipeline останутся локальными в `artifacts/submissions/<SUB-ID>/`; карточка и компактный registry — в `submissions/`.


In [ ]:
saved = submission_tools.save_submission(
    PROJECT_ROOT,
    SUBMISSION_ID,
    prediction,
    dataset_versions=dataset_versions,
    allow_overwrite=True,
)
print('CSV для Kaggle:', saved.submission_path.relative_to(PROJECT_ROOT))
print('Fitted pipeline:', saved.model_path.relative_to(PROJECT_ROOT))
print('Metadata:', saved.metadata_path.relative_to(PROJECT_ROOT))
print('Карточка:', saved.note_path.relative_to(PROJECT_ROOT))
print('Registry:', saved.registry_path.relative_to(PROJECT_ROOT))


## После загрузки на Kaggle

Откройте созданную `SUB-xxx` карточку, заполните поля `kaggle_public_score` / `kaggle_private_score` во frontmatter, дату и короткое наблюдение. Затем выполните следующую sync-ячейку: переобучение не требуется. Kaggle score — внешний сигнал, а не автоматическое решение о champion.


In [ ]:
display(submission_tools.sync_submission_scores(PROJECT_ROOT))
